In [10]:
from torch import nn
import torch

BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

In [11]:
class Encoder(nn.Module):
    def __init__(self, **kwargs):
        super(Encoder, self).__init__(**kwargs)

    def forward(self, X, *args):
        raise NotImplementedError

In [12]:
class RNNEncoder(Encoder):
    def __init__(
        self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0, **kwargs
    ):
        super().__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers, dropout=dropout)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers

    def forward(self, X):
        """
        Args:
            X: shape (batch_size, num_steps) - một batch các câu có độ dài num_steps
        Returns:
            output: shape (num_steps, batch_size, num_hiddens) - đầu ra của RNN ở mỗi bước thời gian
            state: shape (num_layers, batch_size, num_hiddens) - trạng thái ẩn cuối cùng của RNN
        """
        X = self.embedding(X).permute(1, 0, 2)
        # shapeL batch_size, num_steps, embed_size -> num_steps, batch_size, embed_size

        output, state = self.rnn(X)

        return state

## Phân tích

### Bước 1: Nhúng từ (Embedding)
Input: X = (batch_size, num_steps) = (256,50)

Embedding layer:

- $W_{embed}$ = (vocab_size, embed_size) = (10000, 256)
- Output: (batch_size, num_steps, embed_size) = (256, 50, 256)

Permute:
- Output: (num_steps, batch_size, embed_size) = (50, 256, 256)

Lý do: RNN trong Pytorch yêu cầu đầu vào có kích thước (seq_len, batch_size, input_size ~ feature size)

### Bước 2: RNN Forward
- Input: (num_steps, batch_size, embed_size) = (50, 256, 256)
- RNN:
    - rnn: nn.GRU(input_size=embed_size, hidden_size=num_hiddens, num_layers=num_layers)
    - Output shape: (num_steps, batch_size, num_hiddens) = (50, 256, 256)
    - State shape: (num_layers, batch_size, num_hiddens) = (2, 256, 256)


### Shape analysis:

- Input: (batch_size, num_steps) = (256, 50) ~ 256 câu, mỗi câu có 50 từ

Step 0: $X$ = (256, 50) ~ 256 câu, mỗi câu có 50 từ

Step 1: $X_{embed}$ = (256, 50, 256) ~ mỗi từ được nhúng thành vector 256 chiều

Step 2: $X_{emb}$ = permute -> (50, 256, 256) ~ chuẩn bị cho RNN

Step 3:

output, state = rnn(X_emb)

-> output: (50, 256, 256), ~ đầu ra của RNN tại mỗi bước thời gian

state: (2, 256, 256) ~ trạng thái ẩn cuối cùng của RNN sau khi xử lý toàn bộ chuỗi

Return state: (num_layers = 2, batch_size = 256, num_hiddens = 256) = (2, 256, 256)

-> state[0] = (256, 256) ~ trạng thái ẩn của lớp RNN thứ nhất

-> state[1] = (256, 256) ~ trạng thái ẩn của lớp RNN thứ hai được dùng làm đầu vào cho decoder

## Decoder

Encoder và Decoder có thể có số layer khác nhau, hidden size khác nhau -> cần `Wrapper` để quản lý kết nối ~ `EncoderDecoder` base class

Decoder trong `Seq2Seq` nhận: 
- Target tokens đã dịch phải -> dự đoán token tiếp theo
- Encoder state -> thông tin từ câu nguồn để dự đoán câu đích

Decoder trả về: 
1. Logits cho mỗi timestep -> dùng để tính loss (thường là CrossEntropyLoss)
2. Decoder state -> dùng để dự đoán token tiếp theo trong câu đích

In [13]:
class Decoder(nn.Module):
    def __init__(self, **kwargs):
        super(Decoder, self).__init__(**kwargs)
        
    def forward(self, X, state):
        raise NotImplementedError("Subclass must implement forward method")

$x_t = [\text{Embed}(y_{t-1}); c_t]$

- $Embed(y_{t-1})$: vector nhúng của token đích tại thời điểm t-1 có dim = embed_size = 256
- $c_t$: context vector từ encoder (thông tin về câu nguồn) có dim = num_hiddens = 256

-> $x_t$ có dim = embed_size + num_hiddens = 256 + 256 = 512

In [14]:
class RNNDecoder(Decoder):
    def __init__(
        self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0, **kwargs
    ):
        super().__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout
        )

        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        return enc_outputs

    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2)

        context = state[-1].repeat(X.shape[0], 1, 1)

        X_and_context = torch.cat((X, context), dim=2)

        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2).reshape(-1, output.shape[-1])

        return output, state

## Seq2Seq Model

### EncoderDecoder:

In [15]:
class EncoderDecoder(nn.Module):
    """Base class cho toàn bộ Encoder-Decoder model"""
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        """Training forward pass"""
        # Encoder
        enc_state = self.encoder(enc_X)

        # Decoder: khởi tạo state từ encoder outputs
        dec_state = self.decoder.init_state(enc_state, *args)

        # Decoder forward: nhận shifted target
        dec_output, dec_state = self.decoder(dec_X, dec_state)

        return dec_output, dec_state

    def predict(self, prefix, num_steps, vocab, device, bos_token="<bos>", eos_token="<eos>"):
        """Inference: sinh từng token một"""
        self.eval()

        # Lấy BOS id theo cách an toàn, fallback sang token đầu của prefix
        if bos_token in vocab:
            bos_id = vocab[bos_token]
        else:
            bos_id = vocab[prefix[0]]

        outputs = [bos_id]

        # Encode prefix
        X = torch.tensor([[bos_id]], device=device)
        state = self.encoder(X)

        # Decode từng bước
        for _ in range(num_steps):
            state = self.decoder.init_state(state)
            Y = torch.tensor([[outputs[-1]]], device=device)
            pred, state = self.decoder(Y, state)
            pred = pred.argmax(dim=1).item()
            outputs.append(pred)

            if eos_token in vocab and pred == vocab[eos_token]:
                break

        return outputs

### Seq2Seq:

In [16]:
class Seq2SeqEncoder(Encoder):
    """Encoder cho Seq2Seq"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super().__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers,
                          dropout=dropout)

    def forward(self, X):
        embeddings = self.embedding(X)  # (batch, steps, embed)
        embeddings = embeddings.permute(1, 0, 2)  # (steps, batch, embed)
        output, state = self.rnn(embeddings)
        # output: (steps, batch, h)
        # state: (layers, batch, h)
        return state  # Chỉ trả về state cuối


class Seq2SeqDecoder(Decoder):
    """Decoder cho Seq2Seq"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super().__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens,
                          num_layers, dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_state, *args):
        # enc_state: (layers, batch, h)
        # Dùng trực tiếp làm decoder state
        return enc_state

    def forward(self, X, state):
        # X: (batch, steps)
        Y = self.embedding(X)  # (batch, steps, embed)
        Y = Y.permute(1, 0, 2)  # (steps, batch, embed)

        # Context: last layer hidden state
        context = state[-1]  # (batch, h)
        context = context.unsqueeze(0).repeat(Y.shape[0], 1, 1)  # (steps, batch, h)

        # Concatenate target embeddings với context
        Y_and_context = torch.cat((Y, context), -1)  # (steps, batch, embed+h)

        # RNN
        output, state = self.rnn(Y_and_context, state)
        # output: (steps, batch, h)
        # state: (layers, batch, h)

        # Linear projection
        output = self.dense(output)  # (steps, batch, vocab)
        output = output.permute(1, 0, 2)  # (batch, steps, vocab)
        output = output.reshape(-1, output.shape[-1])  # (batch*steps, vocab)

        return output, state

In [19]:
def train_seq2seq(data, net, lr, num_epochs, device, tgt_vocab, bos_token="<bos>"):
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(reduction='none')

    if bos_token not in tgt_vocab:
        raise KeyError(f"Token {bos_token} không tồn tại trong tgt_vocab")
    bos_id = tgt_vocab[bos_token]

    for epoch in range(num_epochs):
        for batch in data:
            X, X_valid_len, Y, Y_valid_len = batch
            X, Y = X.to(device), Y.to(device)

            # Shift right Y cho Decoder input
            # Decoder input: [BOS, y_1, y_2, ..., y_{T-1}]
            # Labels:       [y_1, y_2, ..., y_T, EOS]
            bos = torch.tensor([bos_id] * Y.shape[0],
                              device=device).reshape(-1, 1)
            dec_input = torch.cat([bos, Y[:, :-1]], dim=1)  # (batch, steps)

            # Forward
            Y_hat, _ = net(X, dec_input)

            # Loss
            l = loss_fn(Y_hat, Y.reshape(-1))
            l = l.mean()

            optimizer.zero_grad()
            l.backward()
            grad_clip_val = 1
            nn.utils.clip_grad_norm_(net.parameters(), grad_clip_val)
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}, Loss: {l.item():.4f}")


required_vars = ["src_vocab", "tgt_vocab", "data", "device"]
missing_vars = [name for name in required_vars if name not in globals()]

if missing_vars:
    print(
        "Missing required variables before training: " + ", ".join(missing_vars)
    )
    print(
        "Please run your data-preparation notebook/cells first, then rerun this cell."
    )
else:
    # Khởi tạo
    encoder = Seq2SeqEncoder(
        vocab_size=len(src_vocab),
        embed_size=256,
        num_hiddens=256,
        num_layers=2,
        dropout=0.1,
    )
    decoder = Seq2SeqDecoder(
        vocab_size=len(tgt_vocab),
        embed_size=256,
        num_hiddens=256,
        num_layers=2,
        dropout=0.1,
    )
    net = EncoderDecoder(encoder, decoder)

    train_seq2seq(
        data,
        net,
        lr=0.005,
        num_epochs=100,
        device=device,
        tgt_vocab=tgt_vocab,
        bos_token=BOS_TOKEN,
    )

Missing required variables before training: src_vocab, tgt_vocab, data, device
Please run your data-preparation notebook/cells first, then rerun this cell.
